# EWS Fraud Detection - Tahap 5: Fraud Scoring & Klasifikasi Risiko

Notebook ini menggabungkan penilaian risiko berbasis teori keuangan (Beneish M-Score tanpa DEPI) dan deteksi anomali tanpa pengawasan (Isolation Forest dengan kontaminasi 5%) menjadi satu metrik tunggal **Fraud Score**.

Selain itu, notebook ini juga menggabungkan data rekap perusahaan dari `/Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx` untuk menambahkan kolom nama perusahaan dan sektor pada dataset scored akhir.

### Formula Fraud Score (Skala 0-100):
$$\text{Fraud Score} = 0.40 \times \text{M-Score}_{\text{scaled}} + 0.60 \times \text{Anomaly Score}_{\text{scaled}}$$

### Kategori Risiko:
- **Low**: 0 - 30
- **Medium**: 31 - 60
- **High**: 61 - 80
- **Critical**: > 80

In [2]:
import os
import pandas as pd
import numpy as np

in_path = "Fraud_Dataset_Anomaly.xlsx"
rekap_path = "../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx"
out_path = "Fraud_Dataset_Scored.xlsx"

## 1. Memuat Data & Normalisasi Min-Max

In [3]:
if not os.path.exists(in_path):
    print(f"ERROR: File {in_path} tidak ditemukan!")
else:
    df = pd.read_excel(in_path)
    
    # Min-max scaling untuk M-Score dan Anomaly Score
    m_min, m_max = df['m_score'].min(), df['m_score'].max()
    a_min, a_max = df['anomaly_score_05'].min(), df['anomaly_score_05'].max()
    
    df['m_score_scaled'] = 100 * (df['m_score'] - m_min) / (m_max - m_min)
    df['anomaly_score_scaled'] = 100 * (df['anomaly_score_05'] - a_min) / (a_max - a_min)
    
    print(f"M-Score Range: [{m_min:.4f}, {m_max:.4f}]")
    print(f"Anomaly Score Range: [{a_min:.4f}, {a_max:.4f}]")

M-Score Range: [-10.2521, 13.8732]
Anomaly Score Range: [0.3141, 0.7320]


## 2. Penghitungan Fraud Score & Pengelompokan Kategori Risiko

In [4]:
# Bobot: 40% M-Score + 60% Anomaly Score
df['fraud_score'] = 0.4 * df['m_score_scaled'] + 0.6 * df['anomaly_score_scaled']

# Fungsi pengelompokan risiko
def classify_fraud_status(score):
    if pd.isnull(score):
        return np.nan
    if score <= 30:
        return 'Low'
    elif score <= 60:
        return 'Medium'
    elif score <= 80:
        return 'High'
    else:
        return 'Critical'

df['fraud_status'] = df['fraud_score'].apply(classify_fraud_status)

print("=== Distribusi Kategori Risiko Fraud ===")
print(df['fraud_status'].value_counts(dropna=False))

=== Distribusi Kategori Risiko Fraud ===
fraud_status
Low         1405
Medium       201
High          20
Critical       7
Name: count, dtype: int64


## 3. Penggabungan Data Nama Perusahaan & Sektor

In [5]:
if not os.path.exists(rekap_path):
    print(f"WARNING: File rekap perusahaan di {rekap_path} tidak ditemukan!")
else:
    df_rekap = pd.read_excel(rekap_path)
    
    # Standardisasi join key
    df['kode_clean'] = df['kode'].astype(str).str.strip().str.upper()
    df_rekap['kode_clean'] = df_rekap['Kode'].astype(str).str.strip().str.upper()
    
    # Pembersihan newlines dan trailing spaces pada rekap
    df_rekap['Nama Perusahaan'] = df_rekap['Nama Perusahaan'].astype(str).str.replace(r'[\r\n\t]+', ' ', regex=True).str.strip()
    df_rekap['Sektor'] = df_rekap['Sektor'].astype(str).str.replace(r'[\r\n\t]+', ' ', regex=True).str.strip()
    
    # Left join
    df_rekap_sub = df_rekap[['kode_clean', 'Nama Perusahaan', 'Sektor']].drop_duplicates()
    df = df.merge(df_rekap_sub, on='kode_clean', how='left')
    df.drop(columns=['kode_clean'], inplace=True)
    
    # Susun kolom: Pindahkan Nama Perusahaan dan Sektor ke urutan ketiga dan keempat
    cols = list(df.columns)
    cols.remove('Nama Perusahaan')
    cols.remove('Sektor')
    cols.insert(2, 'Nama Perusahaan')
    cols.insert(3, 'Sektor')
    df = df[cols]
    print("Penggabungan data rekap nama perusahaan & sektor selesai.")

Penggabungan data rekap nama perusahaan & sektor selesai.


## 4. Ekspor Data & Audit Perusahaan Risiko Tertinggi

In [6]:
# Simpan hasil akhir
df.to_excel(out_path, index=False)
print(f"Dataset final dengan Nama Perusahaan & Sektor berhasil disimpan ke '{out_path}'. Shape: {df.shape}")

# Audit 10 Perusahaan dengan Fraud Score Tertinggi
print("\n=== Audit: 10 Perusahaan dengan Fraud Score/Risiko Tertinggi ===")
cols_show = ['kode', 'tahun', 'Nama Perusahaan', 'Sektor', 'fraud_score', 'fraud_status', 'm_score', 'anomaly_score_05']
print(df.sort_values(by='fraud_score', ascending=False)[cols_show].head(10).to_string(index=False))

Dataset final dengan Nama Perusahaan & Sektor berhasil disimpan ke 'Fraud_Dataset_Scored.xlsx'. Shape: (1637, 69)

=== Audit: 10 Perusahaan dengan Fraud Score/Risiko Tertinggi ===
kode  tahun                 Nama Perusahaan                    Sektor  fraud_score fraud_status   m_score  anomaly_score_05
ARGO   2022                 Argo Pantes Tbk        Consumer Cyclicals    99.150113     Critical 13.873175          0.726121
IATA   2022     MNC Energy Investments Tbk.                    Energy    89.689761     Critical  7.654729          0.732041
PUDP   2022          Pudjiadi Prestige Tbk.  Properties & Real Estate    87.151106     Critical  7.655418          0.714349
KOBX   2023          Kobexindo Tractors Tbk               Industrials    84.919430     Critical  7.399425          0.701760
MBSS   2024 Mitrabahtera Segara Sejati Tbk.                    Energy    82.456941     Critical  6.225572          0.698164
CMPP   2022          AirAsia Indonesia Tbk. Transportation & Logistic    80.